In [1]:
import pandas as pd
import torch
from transformers import pipeline
import os
from tqdm import tqdm

# ==============================================================================
# 1. 설정
# ==============================================================================

# 입력 CSV 파일 경로
INPUT_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"

# LLaMA가 변환한 문장이 저장될 출력 CSV 파일 경로
OUTPUT_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama.csv"

# 사용할 LLaMA 모델 ID
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 사용할 장치 자동 설정 (GPU 우선)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_MAP = "auto" if DEVICE == "cuda" else "cpu"

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==============================================================================
# 2. 핵심 기능 함수 정의
# ==============================================================================

def initialize_llama_pipeline(model_id, device_map):
    """LLaMA 텍스트 생성 파이프라인을 초기화합니다."""
    print(f"Initializing LLaMA pipeline for model '{model_id}'...")
    if "cuda" not in str(device_map):
        print("Warning: Running on CPU. This will be very slow.")
    
    try:
        llm_pipe = pipeline(
            "text-generation",
            model=model_id,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device_map=device_map,
        )
        print("LLaMA pipeline initialized successfully.")
        return llm_pipe
    except Exception as e:
        print(f"Error initializing LLaMA pipeline: {e}")
        return None

def rephrase_with_llama(llm_pipe, question: str, option: str) -> str:
    """LLaMA를 사용하여 질문과 보기를 자연스러운 Yes/No 질문으로 변환합니다."""
    system_prompt = """You are an expert prompt engineer. Your task is to convert a question and a multiple-choice option into a single, direct, and natural-sounding yes/no question.
Do not add any explanations or introductory text. Only provide the rephrased question."""

    user_prompt = f"""
Here are some examples:
---
Example 1:
Original Question: What is the man doing in the image?
Option to evaluate: Building sand castles
Rephrased Yes/No Question: Is the man in the image building sand castles?

Example 2:
Original Question: Which of the following foods is not present in the image?
Option to evaluate: Pizza
Rephrased Yes/No Question: Is pizza present in the image?
---
Now, perform the task for the following inputs:
Original Question: {question}
Option to evaluate: {option}
Rephrased Yes/No Question:"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt_str = llm_pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipe(
        prompt_str, max_new_tokens=30, do_sample=True, temperature=0.6, top_p=0.9,
        pad_token_id=llm_pipe.tokenizer.eos_token_id
    )
    
    generated_text = outputs[0]['generated_text'][len(prompt_str):].strip()
    return generated_text.replace('"', '')

In [3]:
# ==============================================================================
# 3. 메인 실행 로직
# ==============================================================================    
# --- 사전 준비: 데모용 CSV 파일 생성 ---
print("Step 0: Preparing sample input CSV...")
sample_csv_data = """ID,Question,A,B,C,D
TEST_000,Which of the following foods is not present in the image?,Pizza,Blueberries,Salmon,Avocado
TEST_001,What is the man in the image doing?,Reading a book,Destroying sand castles,Building sand castles,Swimming in the sea
TEST_002,Where might the family be headed?,To a business meeting,To a school,To a winter ski resort,To a summer vacation spot
"""
if not os.path.exists(INPUT_CSV_PATH):
    with open(INPUT_CSV_PATH, "w", encoding='utf-8') as f:
        f.write(sample_csv_data)
    print(f"Created sample input file: '{INPUT_CSV_PATH}'.")
# --- 사전 준비 끝 ---

# 1. LLaMA 파이프라인 초기화
llm_pipe = initialize_llama_pipeline(MODEL_ID, DEVICE_MAP)
if not llm_pipe:
    print("Failed to initialize LLaMA pipeline. Exiting.")
    exit(1)

# 2. 입력 CSV 파일 로딩
print(f"\nStep 1: Loading data from '{INPUT_CSV_PATH}'...")
try:
    df = pd.read_csv(INPUT_CSV_PATH)
except FileNotFoundError:
    print(f"Error: Input CSV file not found at '{INPUT_CSV_PATH}'.")
    exit(1)

# 3. 결과를 저장할 새로운 컬럼 추가
for col in ['A', 'B', 'C', 'D']:
    df[f'Rephrased_{col}'] = ""

# 4. 각 행과 각 보기에 대해 LLaMA로 문장 변환 실행
print(f"\nStep 2: Rephrasing questions for {len(df)} items using LLaMA...")
# tqdm을 사용하여 진행 상황 시각화
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Rephrasing"):
    question = row['Question']
    for option_key in ['A', 'B', 'C', 'D']:
        option_text = row[option_key]
        
        # 비어있는 옵션은 건너뛰기
        if pd.notna(option_text):
            try:
                rephrased_q = rephrase_with_llama(llm_pipe, question, str(option_text))
                # DataFrame의 특정 셀에 결과 저장
                df.loc[index, f'Rephrased_{option_key}'] = rephrased_q
            except Exception as e:
                print(f"An error occurred while processing row {index}, option {option_key}: {e}")
                df.loc[index, f'Rephrased_{option_key}'] = "Error"

# 5. 최종 결과를 새로운 CSV 파일로 저장
print(f"\nStep 3: Saving rephrased data to '{OUTPUT_CSV_PATH}'...")

# 저장할 컬럼 순서 지정
output_columns = [
    'ID', 'Question', 
    'A', 'Rephrased_A',
    'B', 'Rephrased_B',
    'C', 'Rephrased_C',
    'D', 'Rephrased_D'
]
# 존재하는 컬럼만 선택
final_columns = [col for col in output_columns if col in df.columns]

df.to_csv(OUTPUT_CSV_PATH, index=False, columns=final_columns, encoding='utf-8-sig')
print("Processing complete. New CSV file saved successfully.")

# 6. 결과 확인
print("\n--- Preview of the output CSV ---")
print(df[final_columns].head())


Step 0: Preparing sample input CSV...
Initializing LLaMA pipeline for model 'meta-llama/Meta-Llama-3.1-8B-Instruct'...


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]
Device set to use cuda:0


LLaMA pipeline initialized successfully.

Step 1: Loading data from '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'...

Step 2: Rephrasing questions for 60 items using LLaMA...


Rephrasing: 100%|██████████| 60/60 [01:44<00:00,  1.75s/it]


Step 3: Saving rephrased data to '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama.csv'...
Processing complete. New CSV file saved successfully.

--- Preview of the output CSV ---
         ID                                           Question  \
0  TEST_000  What might be the purpose of the person's work...   
1  TEST_001          What color is the shoe rack in the image?   
2  TEST_002    What might be the deer's behavior in the image?   
3  TEST_003  Considering the equipment used in the image, w...   
4  TEST_004  What is a common ingredient in Gimbap (Korean ...   

                              A  \
0  Building muscle and strength   
1                         Black   
2                        Eating   
3              A Pilates studio   
4                        Cheese   

                                         Rephrased_A  \
0  Is the purpose of the person's workout in the ...   
1               Is the shoe rack in the image black